# Nagel-Schreckenberg Traffic Simulation
## Capstone Project Notebook

Traffic congestion is a daily reality for millions of people, yet its causes are often counter-intuitive. Why do traffic jams sometimes appear on a straight highway with no accident or bottleneck? Why does a single driver tapping their brakes trigger a wave of stop-and-go traffic that persists for hours? These **phantom traffic jams** are an emergent phenomenon — they arise from the collective interaction of many drivers following simple individual rules.

The **Nagel-Schreckenberg (NaSch) model** (1992) is a cellular automaton that captures this behaviour with elegant simplicity. It represents a road as a one-dimensional grid of cells, where each cell is either empty or occupied by a single vehicle. Vehicles have integer speeds and follow four update rules applied simultaneously at each time step: acceleration, slowing down to avoid collisions, random braking (modelling human imperfection), and movement.

Despite its simplicity, the NaSch model reproduces key features of real traffic:

- **Phantom traffic jams** — spontaneous stop-and-go waves that propagate upstream through traffic, caused entirely by the random braking rule
- **Phase transitions** — a sharp transition from free-flowing to congested traffic as density increases beyond a critical threshold
- **The fundamental diagram** — the characteristic relationship between traffic density and flow, with flow peaking at intermediate density and collapsing under congestion
- **Metastability** — at intermediate densities, both free-flow and congested states can coexist

### Traffic Flow Theory

The NaSch model sits within the broader field of **traffic flow theory**, which studies vehicular traffic as a complex system. Three macroscopic quantities describe the state of traffic:

- **Density** ($\rho$) — number of vehicles per unit length (vehicles/km)
- **Flow** ($J$) — number of vehicles passing a point per unit time (vehicles/hour)
- **Average speed** ($\bar{v}$) — mean velocity of vehicles

These are related by the fundamental identity: $J = \rho \cdot \bar{v}$

The **fundamental diagram** ($J$ vs $\rho$) is the central empirical relationship in traffic science. It shows that flow increases linearly at low density (free flow), reaches a maximum at a critical density, and then decreases at higher densities (congestion).

### In this notebook you will:
1. Understand the **Nagel-Schreckenberg** cellular automaton for traffic flow
2. Implement the four update rules step by step
3. Visualise traffic dynamics with **spacetime diagrams**
4. Analyse the **fundamental diagram** (flow vs density)
5. Observe spontaneous **phantom traffic jams**
6. Lay the groundwork for required project extensions

---

## 0 · Setup

We import NumPy for array operations (the road state and vehicle speeds are stored as arrays) and Matplotlib for visualisation. Traffic CA models are naturally parallel — all vehicles update simultaneously — making NumPy's vectorised operations a good fit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

---
## 1 · The Nagel-Schreckenberg Model

The NaSch model (1992) is a **one-dimensional cellular automaton** for single-lane traffic:

- A road of length $L$ cells, periodic boundary (circular road)
- Each cell is either empty or occupied by one vehicle
- Each vehicle has a speed $v \in \{0, 1, \ldots, v_{\max}\}$

### Update Rules (applied simultaneously to all vehicles)

| Step | Rule | Effect |
|------|------|--------|
| 1. **Acceleration** | $v \to \min(v + 1, v_{\max})$ | Drivers speed up if they can |
| 2. **Slowing down** | $v \to \min(v, d - 1)$ | Don't crash into the car ahead ($d$ = gap) |
| 3. **Randomisation** | With probability $p$: $v \to \max(v - 1, 0)$ | Random braking (human imperfection) |
| 4. **Movement** | Advance by $v$ cells | Car moves forward |

The randomisation step is crucial — it causes **spontaneous traffic jams** even without any bottleneck.

### Step 1: Initialise the road

The function `init_road` places `n_cars` vehicles at random positions along a road of `road_length` cells. Each car is assigned a random initial speed between 0 and $v_\text{max}$. Positions are sorted so we can efficiently compute gaps between consecutive vehicles.

The road uses **periodic boundaries** — the last cell connects back to the first, forming a circular track. This eliminates boundary effects and lets us study traffic dynamics in a self-contained system without worrying about inflow/outflow conditions.

In [ ]:
def init_road(road_length, n_cars, v_max, seed=42):
    """Place n_cars randomly on a road of given length.
    
    Returns
    -------
    positions : sorted array of car positions
    speeds    : array of initial speeds (random 0 to v_max)
    """
    rng = np.random.default_rng(seed)
    positions = np.sort(rng.choice(road_length, size=n_cars, replace=False))
    speeds = rng.integers(0, v_max + 1, size=n_cars)
    return positions, speeds

### Step 2: Compute gap to car ahead

The **gap** — the number of empty cells between a car and the one directly ahead — is the critical quantity for the slowing-down rule. A car must not exceed its gap, otherwise it would "crash" into the car ahead.

With periodic boundaries, the last car's gap wraps around to the first car. For example, if car A is at position 90 and car B is at position 5 on a road of length 100, the gap is $5 + 100 - 90 - 1 = 14$ cells. We verify the computation with a simple two-car test case.

In [ ]:
def compute_gaps(positions, road_length):
    """Compute gap (empty cells) between each car and the one ahead.
    Uses periodic boundaries."""
    n = len(positions)
    gaps = np.zeros(n, dtype=int)
    for i in range(n):
        next_pos = positions[(i + 1) % n]
        if i == n - 1:  # wrap around
            next_pos += road_length
        gaps[i] = next_pos - positions[i] - 1
    return gaps

### Step 3: One NaSch update step

The `nasch_step` function applies all four update rules simultaneously to every vehicle. The implementation follows the canonical order:

1. **Acceleration** — every car increases its speed by 1, up to $v_\text{max}$. This models the natural tendency of drivers to speed up when possible.
2. **Slowing down** — each car's speed is capped at the gap to the car ahead. This prevents collisions and is a deterministic safety rule.
3. **Randomisation** — with probability $p_\text{slow}$, each car reduces its speed by 1 (to a minimum of 0). This single stochastic rule is what creates phantom jams: a random brake event propagates backward through dense traffic as each following car must also slow down.
4. **Movement** — each car advances by its current speed. Positions wrap around via modulo arithmetic.

After movement, we re-sort vehicles by position to maintain the ordering needed for gap computation in the next step.

In [ ]:
def nasch_step(positions, speeds, road_length, v_max, p_slow, rng):
    """Apply one Nagel-Schreckenberg update step."""
    n = len(positions)
    gaps = compute_gaps(positions, road_length)
    
    # 1. Acceleration
    speeds = np.minimum(speeds + 1, v_max)
    
    # 2. Slowing down (don't exceed gap)
    speeds = np.minimum(speeds, gaps)
    
    # 3. Randomisation
    brake = rng.random(n) < p_slow
    speeds = np.where(brake, np.maximum(speeds - 1, 0), speeds)
    
    # 4. Movement
    positions = (positions + speeds) % road_length
    
    # Re-sort to maintain order
    order = np.argsort(positions)
    return positions[order], speeds[order]

### Step 4: Full simulation

The `simulate_nasch` function chains initialisation and repeated NaSch steps into a complete simulation. At each time step it records:

- A **spacetime row** — a binary array marking which cells are occupied. Stacking these rows produces the spacetime diagram, the primary diagnostic tool for traffic CA.
- The **speed array** — used to compute average flow and speed statistics.

We run 200 time steps on a road of 200 cells with 60 cars (density $\rho = 0.30$) and $p_\text{slow} = 0.3$. At this density, the system is in the interesting regime where free flow and congestion coexist — phantom jams spontaneously form and dissolve.

In [ ]:
def simulate_nasch(road_length=200, n_cars=60, v_max=5, p_slow=0.3,
                    n_steps=200, seed=42):
    """Run a NaSch simulation. Returns spacetime diagram."""
    rng = np.random.default_rng(seed)
    positions, speeds = init_road(road_length, n_cars, v_max, seed)
    
    spacetime = np.zeros((n_steps, road_length), dtype=int)
    speed_history = []
    
    for t in range(n_steps):
        spacetime[t, positions] = 1
        speed_history.append(speeds.copy())
        positions, speeds = nasch_step(positions, speeds, road_length, v_max, p_slow, rng)
    
    return spacetime, speed_history

st, sh = simulate_nasch()

---
## 2 · Spacetime Diagram

The spacetime diagram is the standard visualisation for traffic CA models. Each row represents the road at a single time step (time increases downward), and each column represents a cell position. Black pixels indicate occupied cells (vehicles).

The key features to look for:
- **Diagonal streaks going up-left** — these are **traffic jams propagating upstream**. Each streak represents a cluster of stopped or slow cars. New cars arrive at the back of the jam and must brake; cars at the front gradually accelerate away. The jam moves backward even though every individual car moves forward.
- **Clear diagonal lines going down-right** — these are individual vehicles in free flow, moving at or near $v_\text{max}$.
- The **width** of the jam streaks indicates how long each car is trapped; the **slope** indicates the jam's propagation speed.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.imshow(st, cmap='binary', aspect='auto', interpolation='nearest')
ax.set_xlabel('Position (cell)')
ax.set_ylabel('Time step')
ax.set_title('Spacetime Diagram — NaSch Traffic Model', fontweight='bold')
plt.tight_layout(); plt.show()

### Animated traffic flow

The spacetime diagram compresses the entire simulation into a single image. The animation below shows the road state evolving in real time — each frame is one time step, with black cells representing vehicles. Watch for phantom jams forming spontaneously and propagating backward (leftward) through traffic while individual cars move forward (rightward).

In [ ]:
# --- Pre-computed spacetime data from st (200 steps x 200 cells) ---
fig, ax = plt.subplots(figsize=(12, 1.5))
road_image = ax.imshow(st[0:1, :], cmap='binary', aspect='auto',
                        interpolation='nearest', vmin=0, vmax=1)
ax.set_xlim(0, st.shape[1])
ax.set_yticks([])
ax.set_xlabel('Position (cell)')

def update_plot(frame_number):
    """Show the road state at one time step."""
    road_image.set_data(st[frame_number:frame_number+1, :])
    ax.set_title(f'NaSch Traffic — Step {frame_number}/{st.shape[0]}')
    return [road_image]

plt.close()

traffic_animation = animation.FuncAnimation(
    fig, update_plot, frames=st.shape[0], interval=80, blit=True
)

HTML(traffic_animation.to_jshtml())

---
## 3 · Effect of Random Braking Probability

The randomisation parameter $p_\text{slow}$ is the single most important control in the NaSch model. It represents the imperfection of human drivers — momentary inattention, over-cautious braking, or delayed reaction to the car ahead accelerating.

Below we compare four values:
- **$p = 0.0$** — perfect drivers, no random braking. Traffic flows freely at all densities below the maximum packing. No jams form.
- **$p = 0.1$** — mild imperfection. Occasional small disturbances, but they dissipate quickly. Jams are rare and short-lived.
- **$p = 0.3$** — moderate imperfection (realistic). Phantom jams form spontaneously and persist, propagating backward through traffic as self-sustaining waves.
- **$p = 0.5$** — highly imperfect drivers. Frequent braking creates dense, persistent jam clusters. Traffic is heavily congested even at moderate densities.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, p_val in zip(axes, [0.0, 0.1, 0.3, 0.5]):
    st_p, _ = simulate_nasch(p_slow=p_val, seed=42)
    ax.imshow(st_p, cmap='binary', aspect='auto', interpolation='nearest')
    ax.set_title(f'p = {p_val}', fontsize=11)
    ax.set_xlabel('Position'); ax.set_ylabel('Time')
fig.suptitle('Effect of Random Braking Probability', fontweight='bold', fontsize=13)
plt.tight_layout(); plt.show()

---
## 4 · Fundamental Diagram

The **fundamental diagram** plots average flow ($J$) vs density ($\rho$). This is the central empirical relationship in traffic science — it characterises the macroscopic behaviour of traffic and is the primary benchmark for any traffic model.

Flow is computed as $J = \rho \cdot \bar{v}$, where $\bar{v}$ is the average speed of all vehicles. We sweep density from near-empty to heavily congested (2%–80% occupancy), running a simulation at each density and averaging over the last 100 steps to exclude transient effects.

The expected shape:
- **Low density** — cars move freely at $v_\text{max}$, so flow increases linearly: $J \approx \rho \cdot v_\text{max}$
- **Critical density** — flow reaches a maximum. Beyond this point, cars begin to interfere with each other.
- **High density** — congestion dominates, average speed drops, and flow decreases despite more cars on the road. This is the "congested branch" of the fundamental diagram.

In [ ]:
road_len = 200
densities = np.linspace(0.02, 0.8, 20)
flows = []

for rho in densities:
    n_cars = max(2, int(rho * road_len))
    _, sh_fd = simulate_nasch(road_length=road_len, n_cars=n_cars,
                               v_max=5, p_slow=0.3, n_steps=300, seed=42)
    # Average flow = density * mean speed (use last 100 steps)
    avg_speed = np.mean([np.mean(s) for s in sh_fd[-100:]])
    flows.append(rho * avg_speed)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(densities, flows, 'bo-', lw=2, markersize=5)
ax.set_xlabel('Density $\\rho$ (cars / cell)')
ax.set_ylabel('Flow $J$ (cars / step)')
ax.set_title('Fundamental Diagram', fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## 5 · Your Tasks

Implement the core NaSch model (done above) and add **at least two** features:

### Task A: Multi-Lane Dynamics
Add a second lane with lane-changing rules: change lane if (1) current lane is blocked, (2) target lane has enough gap, (3) no car approaching fast from behind in target lane.

### Task B: Driver Behaviour Variations
Introduce driver types: aggressive ($p_{\text{slow}}$ low, $v_{\max}$ high), cautious ($p_{\text{slow}}$ high, $v_{\max}$ low), normal. Assign types randomly and compare.

### Task C: Road Network Features
Add a traffic light that alternates red/green every $T$ steps. Vehicles with $v_{\max}$ speed limit in a zone. On-ramp merging.

### Task D: Traffic Monitoring
Place virtual detectors at fixed positions to measure local flow, density, and speed over time. Reproduce the flow-density scatter plot from real traffic data.

### Discussion points
- Show phantom jam formation and dissolution
- Analyse how parameters affect flow
- Compare fundamental diagram to real traffic data
- Discuss limitations of the 1D model

In [ ]:
# TODO: Delete this cell

---
## Recommended Reading & Journal Club

**1. Nagel, K. & Schreckenberg, M. (1992)** *A cellular automaton model for freeway traffic.* Journal de Physique I, 2(12), 2221–2229. [DOI](https://doi.org/10.1051/jp1:1992277)
→ The original NaSch paper.

**2. Chowdhury, D., Santen, L. & Schadschneider, A. (2000)** *Statistical physics of vehicular traffic and some related systems.* Physics Reports, 329(4–6), 199–329. [DOI](https://doi.org/10.1016/S0370-1573(99)00117-9)
→ Comprehensive review of traffic CA models and their physics.

**3. Knospe, W. et al. (2002)** *Single-vehicle data of highway traffic: Microscopic description of traffic phases.* Physical Review E, 65(5), 056133.
→ Connects NaSch-type models to real single-vehicle detector data.

**4. Maerivoet, S. & De Moor, B. (2005)** *Cellular automata models of road traffic.* Physics Reports, 419(1), 1–64. [DOI](https://doi.org/10.1016/j.physrep.2005.08.005)
→ Modern review including multi-lane extensions and on-ramp models.